# Занятие 2:  Проверка гипотез

В этом мини-проекте мы посчитаем несколько продуктовых метрик и посмотрим, какие распределения могут встретиться на практике.

### Описание данных

* ad_id – идентификатор объявления (рекламы)
* xyz_campaign_id – идентификатор рекламной кампании в базе компании X
* fb_campaign_id – идентификатор кампании в Facebook
* age – возрастная группа, которой показывалась реклама
* gender – пол тех, кому показывалась реклама
* interest –код, указывающий категорию, к которой относится интерес пользователя (соответствие число – интерес)
* impressions – число показов
* clicks – число кликов
* spent – сумма, которую компания потратила на показ объявления 
* total_conversion – количество людей, которые заинтересовались продуктом после просмотра рекламы
* approved_conversion – количество людей, которые купили продукт после просмотра рекламы

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as ss

import seaborn as sns
import matplotlib.pyplot as plt

sns.set(rc={'figure.figsize':(12,6)}, style="whitegrid")

In [ ]:
conv = pd.read_csv('https://getfile.dokpub.com/yandex/get/https://disk.yandex.ru/d/Kl4mPINblwnJCA')

In [ ]:
conv.head()

# 1

Загрузите данные, проверьте число наблюдений и столбцов, типы данных, наличие пропущенных значений, какие уникальные значения встречаются.

Сколько уникальных рекламных кампаний было проведено? 

- 3

In [ ]:
conv.shape

In [ ]:
conv.dtypes

In [ ]:
conv.nunique()

In [ ]:
conv.isna().sum()

In [ ]:
conv.xyz_campaign_id.unique()

# 2

Посмотрите на данные и соотнесите переменные с их типом

- Approved_conversion, Total_conversion, Impressions и Clicks - дискретные
- Spent - непрерывная
- age - ранговая
- gender и interest - номинативные

# 3
Постройте график распределения числа показов (Impressions – сколько раз пользователи увидели данное объявление) для каждой рекламы, прологарифмировав значения. Выберите верные утверждения:


- Полученное распределение является нормальным
- **Полученное распределение не является нормальным**
- Распределение унимодальное 
- Распределение симметричное
- **Распределение бимодальное**

In [ ]:
sns.distplot(np.log(conv.groupby('fb_campaign_id').Impressions.sum()), kde=False, bins=50)

In [ ]:
sns.distplot(np.log(conv.query("xyz_campaign_id == 916").groupby('fb_campaign_id').Impressions.sum()))
sns.distplot(np.log(conv.query("xyz_campaign_id == 936").groupby('fb_campaign_id').Impressions.sum()))
sns.distplot(np.log(conv.query("xyz_campaign_id == 1178").groupby('fb_campaign_id').Impressions.sum()))

In [ ]:
#немного кластеризации

from sklearn.mixture import GaussianMixture

dat = np.log(conv.groupby('fb_campaign_id').Impressions.sum()).values.reshape(-1, 1) #готовим данные

mix = GaussianMixture(n_components = 2).fit(dat) #строим модель

labels = mix.predict(dat) #вычисляем принадлежности

In [ ]:
sns.distplot(dat[labels == 0], kde=False, bins=50)
sns.distplot(dat[labels == 1], kde=False, bins=50)

# 4

Теперь посчитаем ещё несколько полезных метрик. Первая – CTR (click-through rate), которая показывает кликабельность, т.е. сколько кликов получила реклама в сравнении с количеством показов. 

$$CTR = \frac{clicks}{impressions}$$

Создайте новую колонку, затем посмотрите на описательные статистики. В качестве ответа укажите ad_id объявления с наибольшим CTR.

In [ ]:
conv['ctr'] = conv.Clicks / conv.Impressions

In [ ]:
conv.ctr.describe()

In [ ]:
conv.set_index("ad_id").ctr.idxmax()

# 5
Визуализируйте CTR с разбивкой по номеру рекламной кампании (xyz_campaign_id). Какому графику соответствует распределение CTR кампании 916?

In [ ]:
sns.displot(data = conv, x="ctr", hue = "xyz_campaign_id", palette = "Accent")

In [ ]:
sns.distplot(conv.query("xyz_campaign_id == 916").ctr, bins=20, kde=False) 

In [ ]:
sns.histplot(conv.query("xyz_campaign_id == 916").ctr, bins=20)

In [ ]:
sns.displot(conv.query("xyz_campaign_id == 916").ctr, bins=20)

# 6
CPC (cost-per-click) – стоимость за клик пользователя по объявлению. Рассчитывается путём деления суммы потраченных денег на общее число кликов:

$$CPC = \frac{spent}{clicks}$$

Выведите описательные статистики для новой переменной, посмотрите на форму распределения. В ответе укажите межквартильный размах.

In [ ]:
conv['cpc'] = (conv.Spent / conv.Clicks)

In [ ]:
conv.cpc.describe()

In [ ]:
round(ss.iqr(conv.cpc, nan_policy='omit'), 2)

# 7

Визуализируйте CPC с разбивкой по полу пользователей, которым были показаны объявления. Какой график получился?

Чтобы избежать появление ошибки, можно указать .dropna() при построении графика. Обратите внимание, что удалять NaN из самого датасета не нужно, только для визуализации. Картинка в полном размере – здесь.



In [ ]:
sns.distplot(conv.query("gender=='M'").cpc.dropna())
sns.distplot(conv.query("gender=='F'").cpc.dropna())

In [ ]:
sns.displot(data=conv, x="cpc", hue="gender")

# 8

Конверсия (conversion rate) – отношение числа пользователей, совершивших целевое действие на определенном этапе, к общему числу тех, кто дошел до данного этапа.

Посчитайте конверсию из клика в покупку. В качестве ответа укажите конверсию для объявления 1121814 в процентах, округлив значение до 2 знаков после точки. Например, если значение кликов равно 10, а покупок – 2, то CR на данном этапе составляет $\frac{2}{10} = 0.2 = 20\%$

In [ ]:
conv['conv_rate'] =((conv.Approved_Conversion / conv.Clicks).mul(100))

In [ ]:
conv.query("ad_id == 1121814").conv_rate.round(2)